# 真实 API：SelfCheck n-gram

使用实际 API 样本运行不需要额外模型的 n-gram 自一致性基线。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai]"`；配置 API key 后会产生调用费用。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `n: int = 1`：n 元语法长度。
- `lowercase: bool = True`：英文大小写归一化。
- `NUM_SAMPLES: int = 5`：真实采样语料规模。
- `TEMPERATURE: float = 0.8`：API 采样多样性。

## 步骤 1：配置真实 API

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
from openai import OpenAI
from hallucination.detection.backends.api import OpenAIResponsesBackend
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.8
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
base_url = os.environ.get("OPENAI_BASE_URL")
client = OpenAI(api_key=api_key, **({"base_url": base_url} if base_url else {}))
backend = OpenAIResponsesBackend(OPENAI_MODEL, client=client, temperature=TEMPERATURE)

## 步骤 2：运行 SelfCheck

In [ ]:
from hallucination.detection.blackbox.selfcheck_ngram import SelfCheckNgramScorer
from hallucination.detection.pipelines.short_answer import BlackBoxSamplingPipeline
result = BlackBoxSamplingPipeline(backend, sentence_scorers=[SelfCheckNgramScorer(n=1, lowercase=True)], num_samples=5).detect_prompt("请用两句话介绍蓝鲸的食性和分类。")
for unit in result.units:
    print(unit.text, unit.method_scores[0].risk, unit.method_scores[0].raw)

## 结果解读与限制

n-gram 只衡量词面重合，对同义改写、否定和事实蕴含不鲁棒，适合作为低成本可解释基线。